# Handwritten digit prediction

Below is the code to train a machine learning model that can predict what digit has been handwritten

The code is structure into two main sections.

First a neural network is trained on the MNIST dataset: https://www.kaggle.com/datasets/hojjatk/mnist-dataset

Second, the inference code so that we can run predictions for incoming data



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import pandas as pd
import json
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## Model Training

Creating and training a multi model neural network that takes images and metadata as input

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 7 * 7, 128)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
class FinalClassifier(nn.Module):
    def __init__(self, image_feat_dim=128, metadata_dim=6, num_classes=10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(image_feat_dim + metadata_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, img_feat, meta_feat):
        return self.fc(torch.cat([img_feat, meta_feat], dim=1))

### Generating meta data for each image

This metadata is being randomly created just for the purpose of this exercise to create a multi model machine learning model.

In [ ]:
def generate_metadata(n):
    rng = np.random.default_rng(42)
    return {
        "pen_pressure": rng.uniform(0.5, 1.5, n),
        "writer_age": rng.integers(10, 80, n),
        "handedness": rng.choice(["left", "right"], n)
    }

### Saving example to run at inference

In [ ]:
def save_example_files(img_array, metadata_row):
    img = Image.fromarray((img_array * 255).astype("uint8"), mode="L")
    img.save("example_digit.png")

    with open("example_metadata.json", "w") as f:
        json.dump(metadata_row, f, indent=2)

### Training the model

In [ ]:
def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)

    images = dataset.data.numpy().astype(np.float32) / 255.0
    images = np.expand_dims(images, 1)
    labels = dataset.targets.numpy()

    metadata_dict = generate_metadata(len(images))
    metadata_df = pd.DataFrame(metadata_dict)

    encoder = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), ["pen_pressure", "writer_age"]),
            ("cat", OneHotEncoder(), ["handedness"])
        ]
    )
    encoder.fit(metadata_df)

    meta_encoded = encoder.transform(metadata_df).astype(np.float32)

    X_img = torch.tensor(images)
    X_meta = torch.tensor(meta_encoded)
    y = torch.tensor(labels, dtype=torch.long)

    image_model = CNNEncoder().to(device)
    final_model = FinalClassifier(metadata_dim=X_meta.shape[1]).to(device)

    optimizer = optim.Adam(
        list(image_model.parameters()) + list(final_model.parameters()),
        lr=1e-3
    )
    loss_fn = nn.CrossEntropyLoss()

    for i in range(0, len(X_img), 128):
        batch_img = X_img[i:i+128].to(device)
        batch_meta = X_meta[i:i+128].to(device)
        batch_y = y[i:i+128].to(device)

        optimizer.zero_grad()
        img_feat = image_model(batch_img)
        logits = final_model(img_feat, batch_meta)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()

        if i % 20000 == 0:
            print(f"Training step {i}, loss={loss.item():.4f}")

    torch.save(image_model.state_dict(), "image_model.pth")
    torch.save(final_model.state_dict(), "final_classifier.pth")
    joblib.dump(encoder, "metadata_encoder.joblib")

    example_img = images[0].squeeze()
    example_meta_row = metadata_df.iloc[0].to_dict()
    save_example_files(example_img, example_meta_row)

    print("✅ Training complete. Model + encoder + example image/metadata saved.")

In [ ]:
train()

## Inference code

Below is the code necessary to make predictions for incoming data

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 7 * 7, 128)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
class FinalClassifier(nn.Module):
    def __init__(self, image_feat_dim=128, metadata_dim=6, num_classes=10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(image_feat_dim + metadata_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, img_feat, meta_feat):
        return self.fc(torch.cat([img_feat, meta_feat], dim=1))

In [ ]:
def load_example_input(image_path, metadata_path):
    img = Image.open(image_path).convert("L")
    img = np.array(img).astype("float32") / 255.0 

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    return img, metadata

In [ ]:
def run_inference(img_array, metadata_dict):
    # Load models
    image_model = CNNEncoder()
    image_model.load_state_dict(torch.load("image_model.pth"))
    image_model.eval()

    metadata_encoder = joblib.load("metadata_encoder.joblib")
    meta_df = pd.DataFrame([metadata_dict])
    metadata_dim = metadata_encoder.transform(meta_df).shape[1]

    # Create model with correct input size
    final_model = FinalClassifier(metadata_dim=metadata_dim)
    final_model.load_state_dict(torch.load("final_classifier.pth"))
    final_model.eval()


    img_tensor = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    meta_df = pd.DataFrame([metadata_dict])
    meta_encoded = metadata_encoder.transform(meta_df).astype("float32")
    meta_tensor = torch.tensor(meta_encoded)

    with torch.no_grad():
        img_feat = image_model(img_tensor)
        logits = final_model(img_feat, meta_tensor)
        pred = torch.argmax(logits, dim=1).item()

    return {"predicted_digit": pred}

### Generate prediction

Run prediction for example generated

In [ ]:
img, meta = load_example_input("example_digit.png", "example_metadata.json")
result = run_inference(img, meta)
print(result)